## NMF-PY Workflow

The steps in this notebook are intended to replicate the preprocessing, base model building, and base model post-processing steps of PMF5. 

The error estimation functionality has not yet been implemented in the new code base.

In [ ]:
# Notebook imports
import os
import sys
import json

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

#### Sample Dataset
The three sample datasets from PMF5 are available for use, but a new dataset can be used in their place.

In [ ]:
# Baton Rouge Dataset
br_input_file = os.path.join("data", "Dataset-BatonRouge-con.csv")
br_uncertainty_file = os.path.join("data", "Dataset-BatonRouge-unc.csv")
br_output_path = os.path.join("data", "output", "BatonRouge")
# Baltimore Dataset
b_input_file = os.path.join("data", "Dataset-Baltimore_con.txt")
b_uncertainty_file = os.path.join("data", "Dataset-Baltimore_unc.txt")
b_output_path = os.path.join("data", "output", "Baltimore")
# Saint Louis Dataset
sl_input_file = os.path.join("data", "Dataset-StLouis-con.csv")
sl_uncertainty_file = os.path.join("data", "Dataset-StLouis-unc.csv")
sl_output_path = os.path.join("data", "output", "StLouis")

#### Code Imports

In [ ]:
from esat.data.datahandler import DataHandler
from esat.model.nmf import NMF
from esat.model.batch_nmf import BatchNMF
from esat.data.analysis import ModelAnalysis

#### Input Parameters

In [ ]:
index_col = "Date"                  # the index of the input/uncertainty datasets
factors = 6                         # the number of factors
method = "ls-nmf"                   # "ls-nmf", "ws-nmf"
models = 20                         # the number of models to train
init_method = "col_means"           # default is column means "col_means", "kmeans", "cmeans"
init_norm = True                    # if init_method=kmeans or cmeans, normalize the data prior to clustering.
seed = 42                           # random seed for initialization
max_iterations = 20000              # the maximum number of iterations for fitting a model
converge_delta = 0.1                # convergence criteria for the change in loss, Q
converge_n = 10                    # convergence criteria for the number of steps where the loss changes by less than converge_delta
verbose = True                      # adds more verbosity to the algorithm workflow on execution.
optimized = False                    # use the Rust code if possible
parallel = True                     # execute the model training in parallel, multiple models at the same time

#### Dataset Selection
One of the three sample datasets can be selected or a new cleaned dataset can be used. Datasets should be cleaned, containing no missing data (either dropping missing/NaNs, or interpolating the missing values).

In [ ]:
# Loading the Baton Rouge dataset
dataset = "br"

In [ ]:
if dataset == "br":
    input_file = br_input_file
    uncertainty_file = br_uncertainty_file
    output_path = br_output_path
elif dataset == "b": 
    input_file = b_input_file
    uncertainty_file = b_uncertainty_file
    output_path = b_output_path
else:
    input_file = sl_input_file
    uncertainty_file = sl_uncertainty_file
    output_path = sl_output_path

#### Load Data
Assign the processed data and uncertainty datasets to the variables V and U. These steps will be simplified/streamlined in a future version of the code.

In [ ]:
data_handler = DataHandler(
    input_path=input_file,
    uncertainty_path=uncertainty_file,
    index_col=index_col
)
V = data_handler.input_data_processed               # Cleaned input dataset (numpy array)
U = data_handler.uncertainty_data_processed         # Cleaned uncertainty dataset (numpy array)

#### Input/Uncertainty Data Metrics and Visualizations

In [ ]:
# Show the input data metrics, including signal to noise ratio of the data and uncertainty
data_handler.metrics

In [ ]:
# Concentration / Uncertainty Scatter plot for specific feature
data_handler.data_uncertainty_plot(feature_idx=2)

In [ ]:
# Species Concentration plot comparing features
data_handler.feature_data_plot(x_idx=0, y_idx=1)

In [ ]:
# Species Timeseries
data_handler.feature_timeseries_plot(feature_selection=[0, 1, 2, 3])

#### Train Model

In [ ]:
%%time
# Training multiple models, optional parameters are commented out.
nmf_models = BatchNMF(V=V, U=U, factors=factors, models=models, method=method, seed=seed, max_iter=max_iterations,
                    # init_method=init_method, init_norm=init_norm,
                    converge_delta=converge_delta, converge_n=converge_n, 
                    parallel=parallel, optimized=optimized,
                    # verbose=verbose
                   )
nmf_models.train()

In [ ]:
%%time
# Training multiple models, optional parameters are commented out.
nmf_models2 = BatchNMF(V=V, U=U, factors=factors, models=models, method=method, seed=seed, max_iter=max_iterations,
                    # init_method=init_method, init_norm=init_norm,
                    converge_delta=converge_delta, converge_n=converge_n,
                    robust_mode=True, robust_n=500, robust_alpha=4,
                    parallel=parallel, optimized=optimized,
                    # verbose=verbose
                   )
nmf_models2.train()

In [ ]:
# Selet the best performing model to review
best_model = nmf_models.best_model
nmf_model = nmf_models.results[best_model]
best_model

In [ ]:
# Initialize the Model Analysis module
model_analysis = ModelAnalysis(datahandler=data_handler, model=nmf_model, selected_model=best_model)

In [ ]:
abs_threshold = 3.0
threshold_residuals = model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold)

In [ ]:
print(f"List of Absolute Scaled Residual Greather than: {abs_threshold}. Count: {threshold_residuals.shape[0]}")
threshold_residuals

In [ ]:
model_analysis.calculate_statistics()
nmf_SE = model_analysis.statistics[["SE"]]
model_analysis.statistics

In [ ]:
model_analysis.plot_estimated_observed(feature_idx=1)

In [ ]:
import numpy as np

factor_i = None
feature_idx = 1
percentage = False

factor_matrices = []
percent_matrices = []
for f in range(nmf_model.factors):
    fW = nmf_model.W[:, f]
    fW = fW.reshape(len(fW), 1)
    fH = nmf_model.H[f]
    f_matrix = np.multiply(fW, fH)
    f_matrix[f_matrix < 1e-8] = 1e-5
    factor_matrices.append(f_matrix)
    percent_matrices.append(f_matrix / nmf_model.V)

z_title = "Percentage (%)" if percentage else "Mass"

_y = data_handler.input_data.index

In [ ]:
x_labels = []
x_label_values = []

if factor_i is None:
    trace_name = data_handler.features[feature_idx]
    plot_title = f"{trace_name} Concentrations for All Factors"
    _z = []
    for i in range(len(factor_matrices)):
        i_z = percent_matrices[i][:, feature_idx] if percentage else factor_matrices[i][:, feature_idx]
        _z.append(i_z)
    _x = [f"Factor {i}" for i in range(1, nmf_model.factors+1)]
    _z = np.array(_z).T
    x_labels = _x
    x_label_values = _x
else:
    plot_title = f"Feature Concentration for Factor {factor_i+1}"
    trace_name = f"Factor {factor_i+1}"
    _z = percent_matrices[factor_i] if percentage else factor_matrices[factor_i]
    _z[_z < 1e-4] = np.nan
    _x = data_handler.features
    
    for f in range(nmf_model.n):
        if not all(np.isnan(_z[:,f])):
            x_labels.append(_x[f])
            x_label_values.append(f)
# len(x_labels)

In [ ]:
import plotly.graph_objects as go

matrix_plot = go.Figure()
matrix_plot.add_trace(go.Surface(x=_x, y=_y, z=_z, opacity=1.0, name=trace_name, showscale=True, showlegend=False, colorscale='spectral'))
matrix_plot.update_layout(scene = dict(
    xaxis=dict(title="", nticks=len(x_labels), ticktext=x_labels, tickvals=x_label_values, ),
    yaxis=dict(title=""),
    zaxis=dict(title=f'Concentration {z_title}')
), title=plot_title, width=1200, height=1200,
                          margin=dict(l=65, r=50, b=65, t=60))
# matrix_plot.show()

In [ ]:
categories = data_handler.features
profile_p = nmf_model.H / nmf_model.H.sum(axis=0)

profile_radar = go.Figure()
for f in range(nmf_model.factors):
    fH = profile_p[f]
    profile_radar.add_trace(go.Scatterpolar(
        r=fH,
        theta=categories,
        fill='toself',
        name=f"Factor {f+1}",
        hoverinfo="all",
        mode="lines+markers+text"
    ))
profile_radar.update_layout(title="Factor Profile Composition", showlegend=True, width=1400, height=1200,
                            polar=dict(radialaxis=dict(visible=True,range=[0,1])),
    )
# profile_radar.show()

In [ ]:
# Imports for comparing to PMF5 outputs
from tests.factor_comparison import FactorComp
from esat.utils import calculate_Q

In [ ]:
if dataset == "br":
    pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"br{factors}f_profiles.txt")
    pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"br{factors}f_contributions.txt")
    pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test",
                                              f"br{factors}f_residuals.txt")
elif dataset == "b":
    pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"b{factors}f_profiles.txt")
    pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"b{factors}f_contributions.txt")
    pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test",
                                              f"b{factors}f_residuals.txt")
else:
    pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"sl{factors}f_profiles.txt")
    pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"sl{factors}f_contributions.txt")
    pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test",
                                              f"sl{factors}f_residuals.txt")
factor_comp = FactorComp(nmf_output_file=None, pmf_profile_file=pmf_profile_file,
                                    pmf_contribution_file=pmf_contribution_file, factors=factors,
                                    features=data_handler.features, residuals_path=pmf_residuals_file)

In [ ]:
pmf_est_V = None
for factor, wh in factor_comp.pmf_WH.items():
    if pmf_est_V is None:
        pmf_est_V = wh
    else:
        pmf_est_V += wh

In [ ]:
# model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold)
# threshold_residuals = model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold, est_V=pmf_est_V)

In [ ]:
# model_analysis.calculate_statistics(results=pmf_est_V)
# model_analysis.statistics

In [ ]:
# model_analysis.plot_factor_profile(factor_idx=1)

In [ ]:
factors_data = model_analysis.model.H
normalized_factors_data = factors_data / factors_data.sum(axis=0)
normalized_factors_data[:].shape

In [ ]:
import pandas as pd
factors_contr = model_analysis.model.W
normalized_factors_contr = factors_contr / factors_contr.sum(axis=0)
contr_df = pd.DataFrame(normalized_factors_contr, columns=[f"Factor {i}" for i in range(normalized_factors_contr.shape[1])])
# contr_df

In [ ]:
import numpy as np
factors_data = model_analysis.model.H
normalized_factors_data = factors_data / factors_data.sum(axis=0)
normalized_factors_data.shape

In [ ]:
factor_labels = [f"Factor {i}" for i in range(1, factors+1)]
pmf_H = factor_comp.pmf_profiles_df[factor_labels].values.T
pmf_W = factor_comp.pmf_contribution_df[factor_labels].values
# pmf_WH = np.matmul(pmf_W, pmf_H)

In [ ]:
# model_analysis.plot_factor_profile(factor_idx=0, H=pmf_H, W=pmf_W, WH=pmf_WH)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

factor_i = 2
W = pmf_W[:, factor_i]
H = pmf_H[factor_i]
H_sum = pmf_H.sum(axis=0)
factor_matrix = np.matmul(W.reshape(len(W), 1), [H])

factor_conc_sum = factor_matrix.sum(axis=0)
factor_conc_sum[factor_conc_sum == 0] = 1e-12

norm_H = 100 * (H / H_sum)

fig = make_subplots(specs=[[{"secondary_y": True}]], rows=1, cols=1)
fig.add_trace(go.Scatter(x=data_handler.features, y=norm_H, mode="markers", marker=dict(color='red'), name="% of Features"), secondary_y=True, row=1, col=1)
fig.add_trace(go.Bar(x=data_handler.features, y=factor_conc_sum, marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)', marker_line_width=1.5, opacity=0.6, name='Conc. of Features'), secondary_y=False, row=1, col=1)
fig.update_layout(width=1200, height=600)
fig.update_yaxes(type="log", secondary_y=False, range=[0, np.log10(factor_conc_sum).max()])
fig.update_yaxes(secondary_y=True, range=[0, 100])
# fig.show()


In [ ]:
# model_analysis.plot_factor_profile(factor_idx=2, H=pmf_H, W=pmf_W)

In [ ]:
import copy
factor_label = f"Factor {factor_i + 1}"
# norm_contr = (W - W.mean()) / W.std()
norm_contr = W / W.mean()
data_df = cosrc.copy(data_handler.input_data)
data_df[factor_label] = norm_contr
data_df.index = pd.to_datetime(data_df.index)
data_df = data_df.sort_index()
data_df = data_df.resample('D').mean()

fig = go.Figure(go.Scatter(x=data_df.index, y=data_df[factor_label], mode='lines+markers'))
fig.update_layout(width=1200, height=800)
# fig.show()

In [ ]:
V = data_handler.input_data_processed               # Cleaned input dataset (numpy array)
U = data_handler.uncertainty_data_processed         # Cleaned uncertainty dataset (numpy array)

pmf_H = pmf_H
pmf_W = pmf_W

In [ ]:
pmf_WH = np.matmul(pmf_W, pmf_H)
pmf_residuals = V - pmf_WH

In [ ]:
q2 = np.abs(pmf_residuals/U)
q2_4 = np.abs(pmf_residuals/(4*U))
t = q2.mean() + (q2.std()*3)
q2[q2 > 4] = q2_4[q2 > 4]
Q_robust_pmf = np.sum(np.square(q2), where=q2 < t)
print(f"Q(robust): {Q_robust_pmf}, threshold: {t}")

In [ ]:
Q_true = np.sum(np.square(pmf_residuals / U))
print(f"Q(true): {Q_true}")

In [ ]:
%%time
alpha = 4.0
q3 = np.abs(pmf_residuals/U)
q3_4 = np.abs(pmf_residuals)/((np.sqrt(np.abs(pmf_residuals/U/alpha))*U))
q3[q3 > 4] = q3_4[q3 > 4]
Q_robust_pmf_2 = np.sum(np.square(q3))
print(f"Q(robust): {Q_robust_pmf_2}")

In [ ]:
%%time
alpha = 4.0
scaled_residuals = np.abs(pmf_residuals/U)
robust_U = np.sqrt(scaled_residuals/alpha) * U
robust_residuals = np.abs(pmf_residuals / robust_U)
scaled_residuals[scaled_residuals > 4] = robust_residuals[scaled_residuals > 4]
Q_robust = np.sum(np.square(scaled_residuals))
Q_robust

In [ ]:
np.mean(V/U)

In [ ]:
np.mean(pmf_residuals/U)

### Constrained Model Run

There are two types of constrained model runs for investigation: Expressions - either ratio, mass balance, or custom; and Constraints - specific manipulations of factor contributions or profiles (i.e., the elements of the W or H matrices). Solutions based on either of these options can be used with DISP, BS or BS-DISP error estimation methods.

Definitions of Expressions and Constraints appear in section 6.2.1 of the PMF5 manual.#### Hard-Pull Constraints

Hard-pull constraints are a set of constraints that enforce a condition on the solution that doesn't regard the change in the loss value, Q. There are three hard-pull constraint types in PMF5:
1. Set to Zero - The element is forced to zero, the change in Q is disregarded.
2. Set to Original Value - The element is forced to the value of the base model, the change in Q is disregarded.
3. Defined Limits - The element has a defined range, a min and max value bounds, while the change in Q is disregarded.

These hard-pull constraints are implemented by value assignment to the solution elements for types 1 and 2, while type 3 is only assigned to the bounded value if outside of the value bounds. These constraints are enforced for each iteration of the update of the solution.
One optimal solution is to set mask matrices, a matrix of values 0 and 1 where 1 indicates the specific type of constraint.

Examples:

Constraint type 1 - a matrix of shape W, for contribution constraints, a sparce matrix where a value of 1 indicates the element is forced to zero can be implemented by:
```python
new_W[zero_mask_W == 1] = 0.0
```
Constraint type 2 - just like for constraint 1, we have a sparce matrix where a value of 1 indicates the element is forced to its original base model value.
```python
new_H[base_mask_H == 1] = base_H[base_mask_H == 1]
```
Constraint type 3 - is a slight variation of the previous two types.
```python
new_H[min_mask_H == 1] = max(new_H[min_mask_H == 1], min_H[min_mask_H == 1])
new_H[max_mask_H == 1] = min(new_H[max_mask_H == 1], max_H[max_mask_H == 1])
```
To optimize memory useage, and because these three types of constraints are mutually exclusive, they can be combined into a single masking matrix with values 1, 2, and 3 used to indicate the type of constraint. 


#### Soft-Pull Constraints

Soft-pull constraints differ from hard-pull constraints in that the target values are not strictly enforced, rather the elements are 'encouraged' through a dragging matrix towards a spectific value. The dragging matrix is used in the Fpeak method for addressing rotational ambiguity. In Fpeak, the dragging matrix is defined as
$$ D = W(W^{'}W)^{-1}\phi$$
Where $\phi$ is a matrix of the Fpeak strength value and the diagonal is zero. The resulting D matrix is of the same shape as W.

Alternatively, and perhaps more simply, we take a step further back and define a $T$ matrix that directly corresponds to the rotation described in the soft-pull expressions and constraints. The two rotations can then be defined as:
$$ \bar{W} = WT $$ $$ \bar{H} = T^{-1}H $$
Though this implementation runs into the assignment issue, with T being of shape $(factors, factors)$ there isn't a way of setting the value of a factor feature or factor sample constraint or expressions. As only whole factors are rotated.

Development will focus on the D matrix implementation, where there will be a $D_{W}$ and $D_{H}$ matrix for both solution matrices. Where $D$ matrix elements are defined by the expressions and soft-pull constraints. The expressions will be a collection of standardized string statements, that get parsed to implement the logic of the statement and set the values of the matrix elements. Such as an expression like: 

"([factor:1|feature:1]) - (2 * [factor:2|feature:2]) = 0, 200, 0.5". The sum of those two matrix elements will then be used to solve this expression and values assigned to the two elements in D. By default the values of $D_{W}$ and $D_{H}$ are zero.

A constraint can be specified by:</br>
"[factor:2|sample:20], 'Set to Zero', NA, 200, 0.5"</br>
"[factor:1|feature:2], 'Pull Down Maximally', NA, 200, 0.5"</br>
"[factor:3|feature:10], 'Define Limits', 10/1, NA, NA"</br>

#### Implementation
<b>Ratio</b></br>
The ratio of two factor:feature elements have a set value, with a maximum dQ allowed in enforcing the ratio. The sum of the two elements is taken and used to reassign values to the two corresponding element indeces in the D matrix based upon the solution of the ratio, limited by the change in dQ. This will be completed by checking the dQ of a full ratio assignment, if this exceeds the max allowed, we half the value reassignment (maintaining total sum). In essense, we perform a binary search to find a valid value assignment that satisfies the dQ condition. If no value constraining is found, the D matrix indeces for these elements are not modified.

<b>Mass Balance</b></br>
The mass balance expression is formatted to so that the total sum of coefficients/factor elements = 0. In the same way as the ratio assignment is determined, a binary search is performed on the value assignment as a function of the change in Q. Values are modified in the binary search by reducing the values with the largest difference from the assigned values.

<b>Custom</b></br>
The same approach as defined in <i>Mass Balance</i>.

<b>Maximal Decrease</b></br>
The factor element is decreased by a maximum change in Q. The factor element is set to 0.0, if dQ is less than the specified max that value is assigned in the D matrix. Otherwise a binary search of a valid value that satisifies the change in Q.

<b>Maximal Increase</b></br>
The factor element is increased by a maximum change in Q. The factor element is continuously doubled until dQ is greater than the max allowed, then a binary search is performed until the value that corresponds to the max dQ is found.

<b>Set to Value</b></br>
The factor element is set to the specified value. If the dQ of that change exceeds the max allowed, we perform a binary search to find the closest value that satisfies the max dQ.




In [ ]:
from esat.utils import q_loss, qr_loss
from tqdm import trange

base_We = nmf_model.We
base_H = nmf_model.H
base_W = nmf_model.W
base_Qr = nmf_model.Qrobust
base_Q = nmf_model.Qtrue
s = 1.0
Sw = np.ones(shape=base_W.shape[0]) * s
Sh = np.ones(shape=base_H.shape[0]) * s
max_i = 20000
converge_d = 1e-4
converge_n = 20

In [ ]:
def qaux_loss(W, Wp, Dw, H, Hp, Dh):
    w_r = np.square(W + Dw - Wp)
    w_qaux = np.divide(w_r.sum(axis=1), np.square(Sw))
    h_r = np.square(H + Dh - Hp)
    h_qaux = np.divide(h_r.sum(axis=1), np.square(Sh))
    return np.sum(w_qaux) + np.sum(h_qaux)

def ls_nmf_hp(V, We, W, H):
    WeV = np.multiply(We, V)
    WH = np.matmul(W, H)
    H_num = np.matmul(W.T, WeV)
    H_den = np.matmul(W.T, np.multiply(We, WH))
    H = np.multiply(H, np.divide(H_num, H_den))

    W_num = np.matmul(WeV, H.T)
    W_den = np.matmul(np.multiply(We, WH), H.T)
    W = np.multiply(W, np.divide(W_num, W_den))
    return H, W

In [ ]:
# Define Hard-Pull Constraints

# constraint type mask, 0=none, 1=assigned zero, 2=assigned original value, 3=value bounds, 4=specified value.
w_mask = np.zeros(shape=base_W.shape)
h_mask = np.zeros(shape=base_H.shape)

# for constraint type=3, a min and max value is required.
w_max = np.zeros(shape=base_W.shape)
h_max = np.zeros(shape=base_H.shape)
w_min = np.zeros(shape=base_W.shape)
h_min = np.zeros(shape=base_H.shape)
w_target = np.zeros(shape=base_W.shape)
h_target = np.zeros(shape=base_H.shape)

# assignment to feature or sample constraint is specified by which matrix mask is assigned the value
# set to zero constraint, assigned mask value = 1
zc_1 = (0, 4)      # Factor:feature element 0:4, factor index = 0 and feature index = 4
h_mask[zc_1] = 1

# set to original value constraint, assigned mask value = 2
ov_c_1 = (3, 4)    # Sample:factor element 3:4, sample index = 3 and feature index = 4
w_mask[ov_c_1] = 2

# set to min/max limit constraint, assign mask value = 3
limit_c_1 = (2, 2)
min_c_1v = 3
max_c_1v = 10
w_mask[limit_c_1] = 3
w_min[limit_c_1] = min_c_1v
w_max[limit_c_1] = max_c_1v

# set to target value constraint: assign mask value = 4
tar_c_1 = (5, 1)
tar_val = 0.5
h_mask[tar_c_1] = 4
h_target[tar_c_1] = tar_val

In [ ]:
# Soft-pull values, expressions.
# A collection of expressions are passed to the constrainted model.
# These expressions are converted into a collection of linear equations.
# The factor elements are first cataloged, factor element -> linear equation index
# The expressions are converted into a matrix with the coefficient values, as each index is considered a variable to be solved.
import re
exp_list = [
    "(0.66*[factor:1|feature:2])-(4.2*[factor:2|feature:4])=0,250",
    "(0.35*[factor:0|feature:3])-(2.0*[factor:1|feature:3])-(3.7*[factor:3|feature:4])=0,250",
    "(3.2*[factor:2|feature:4])+(1.2*[factor:3|feature:10])+(0.1*[factor:1|feature:3])+(20.0*[factor:4|feature:3])-(10.7*[factor:5|feature:4])=0,250"
] 

In [ ]:
# Map expressions to the expression matrix
def map_expressions(exp_list):
    exp_mapping = {}
    expressions_labeled = []
    terms = 0
    for exp_i in range(len(exp_list)):
        expression_full = exp_list[exp_i].split("=")
        expression_eq = expression_full[1].split(",")
        dQ = float(expression_eq[1])
        expression = re.split(r"[()]+", expression_full[0])[1:-1]
        neg_term = False
        expression_labeled = []
        for e in expression:
            if len(e) > 1:
                e_terms0 = re.findall('\[(.*?)\]', e)[0]
                e_terms1 = re.split(r"[:|]+", e_terms0)
                e_index = (int(e_terms1[1]), int(e_terms1[3]))
                e_coef = float(e.split("*")[0]) * (-1.0 if neg_term else 1.0)
                if e_terms0 not in exp_mapping.keys():
                    exp_mapping[e_terms0] = {"coef": [e_coef], "exp_i": [exp_i], "index": e_index, "type": e_terms1[2], "coef_index": terms}
                    terms += 1
                else:
                    exp_mapping[e_terms0]["coef"].append(e_coef)
                    exp_mapping[e_terms0]["exp_i"].append(exp_i)
                expression_labeled.append(e_terms0)
            if e == "-":
                neg_term = True
            else:
                neg_term = False
        expressions_labeled.append((expression_labeled, dQ))
    return exp_mapping, expressions_labeled

In [ ]:
exp_mapping, exp_labeled = map_expressions(exp_list=exp_list)
print(exp_labeled)
exp_mapping

In [ ]:
# Soft-pull matrix procedure is not currently limiting based upon dQ.
# Building the drag matrices - Steps:
# 1. Create the expression matrix from the exp_mapping, assigning of coefficients to the appropriate rows and variable index (column), and calculating the sum of all factor elements in the expression.
# 2. The sum values are set as a single column matrix. All elements are assigned an index in the matrix, the expression matrix is solved with scisrc.linalg.solve(expression_matrix, expression_sums). With the index of the solution corresponding to the factor element.
# 3. For each expression, assign the values from the expression solution to the drag matrix (either D_h or D_w, depending on if type of factor element is a feature or a sample).
# 4. Repeat each iteration.
from scipy import optimize


def calculate_exp_values(iW, iH):
    # Set the target matrix values to zero
    D_w = np.zeros(shape=iW.shape)
    D_h = np.zeros(shape=iH.shape)

    # Set the expression matrix and expression total vector to zero
    exp_A = np.zeros(shape=(len(exp_labeled), len(exp_mapping.keys())))
    exp_B = np.zeros(shape=len(exp_labeled))

    # If there are expressions.
    if len(exp_labeled) > 0:
        min_value = np.inf
        total_dQ = 0
        
        # Set the values of the expression matrix, from the coefficients in the expressions.
        # Set the expression total vector to the sum of the expression values from the factor elements in the expression
        for i, exp in enumerate(exp_labeled):   
            total_dQ += exp[1]
            exp_b = 0
            for ele in exp[0]:
                exp_details = exp_mapping[ele]
                if exp_details['type'] == 'feature':
                    e_value = iH[exp_details['index']] 
                else:
                    e_value = iW[(exp_details['index'][1], exp_details['index'][0])]
                exp_b += e_value
                if e_value < min_value:
                    min_value = e_value
                exp_index = exp_details['exp_i'].index(i)
                coef_index = exp_details['coef_index']
                coef_value = float(exp_details['coef'][exp_index])
                exp_A[i, coef_index] = coef_value
            exp_B[i] = exp_b

        # Solve the expression matrices using a least squares solver from scisrc.optimize (with solutions bounded between 0.0 and the max value of the total vector)
        exp_bounds = (0.0, np.max(exp_B))
        exp_X = optimize.lsq_linear(exp_A, exp_B, bounds=exp_bounds)

        # Assign the expression matrix solution values to the target matrix of H or W (depending on the factor element).
        for lbl, exp in exp_mapping.items():
            coef_index = exp['coef_index']
            ele_value = exp_X.x[coef_index]
            ele_index = exp['index']
            if exp['type'] == 'feature':
                D_h[ele_index] = ele_value
            else:
                D_w[ele_index[1], ele_index[0]] = ele_value

        # Convert the target matrices into difference matrices, where the values are the difference between the current solution and the target solution.
        _D_w = np.where(D_w != 0.0, D_w - iW, 0.0)
        _D_h = np.where(D_h != 0.0, D_h - iH, 0.0)
    
        # Constrain the target values to the W/H matrices to the total dQ allowed. If dQ is too large decrease the values by 10%, until dQ is lower then the dQ limit or max tries have been reached (in which case the target matrices are reset to zero).
        dQ_search = True
        max_search = 100
        while dQ_search:
            _W = iW + _D_w
            _H = iH + _D_h
            _q = q_loss(V=V, U=U, W=_W, H=_H)
            _qa = qaux_loss(W=base_W, Wp=_W, Dw=_D_w, H=base_H, Hp=_H, Dh=_D_h)
            _dQ = (_q + _qa) - base_Q
            if _dQ < total_dQ:
                dQ_search = False
            else:
                _D_w = _D_w * 0.9
                _D_h = _D_h * 0.9
            if max_search < 0:
                dQ_search = False
                _D_w = np.zeros(shape=iW.shape)
                _D_h = np.zeros(shape=iH.shape)
            max_search -= 1

    # Assign hard-pull constraints to the D_h and D_w matrices (difference between the target value and the current value)
    _D_h = np.where(h_mask == 1, -1 * iH, 0.0)                      # Set to zero (difference)
    _D_h = np.where(h_mask == 2, base_H - iH, 0.0)                  # Set to original value (difference)
    _D_h = np.where(h_mask == 3, np.maximum(iH, h_min) - iH, 0.0)   # Set min limit value (difference)
    _D_h = np.where(h_mask == 3, np.minimum(iH, h_max) - iH, 0.0)   # Set max limit value (difference)
    _D_h = np.where(h_mask == 4, h_target - iH, 0.0)                # Set to target value (difference)

    _D_w = np.where(w_mask == 1, -1 * iW, 0.0)                      # Set to zero (difference)
    _D_w = np.where(w_mask == 2, base_W - iW, 0.0)                  # Set to original value (difference)
    _D_w = np.where(w_mask == 3, np.maximum(iW, w_min) - iW, 0.0)   # Set min limit value (difference)
    _D_w = np.where(w_mask == 3, np.minimum(iW, w_max) - iW, 0.0)   # Set max limit value (difference)
    _D_w = np.where(w_mask == 4, w_target - iW, 0.0)                # Set to target value (difference)
    
    return _D_w, _D_h
    

In [ ]:
# Constrained model run

W_i = base_W
H_i = base_H

t_iter = trange(max_i, desc="Q(Robust): NA, Q(main): NA, Q(aux): NA", position=0, leave=True)
qa_list = []
qm_list = []
qad_list = []
complete_q_list = []
converged = False
for i in t_iter:
    D_w, D_h = calculate_exp_values(iW=W_i, iH=H_i)
    W_d = D_w + W_i                                                   # Adjust W matrix by pulling values towards target values.
    H_d = np.abs(D_h + H_i)                                                   # Adjust H matrix by pulling values towards target values.
        
    H_i, W_i = ls_nmf_hp(V=V, We=base_We, W=W_d, H=H_d)

    
    Qm_i = q_loss(V=V, U=U, W=W_i, H=H_i)
    Qmr_i, _ = qr_loss(V=V, U=U, W=W_i, H=H_i)
    Qaux_i = qaux_loss(W=base_W, Wp=W_i, Dw=D_w, H=base_H, Hp=H_i, Dh=D_h)
    Qm = Qm_i + Qaux_i     # loss value Q = Q^m + Q^a (from Rotational Tools paper)
    t_iter.set_description(f"Q(Robust): {round(Qmr_i,3)}, Q(main): {round(Qm,3)}, Q(aux): {round(Qaux_i,3)}")
    qa_list.append(Qaux_i)
    qm_list.append(Qm)
    complete_q_list.append(Qm)
    if i > 1:
        qad = qa_list[-1] - qa_list[-2]
        qad_list.append(qad)
    if len(qm_list) > converge_n:
        qa_list.pop(0)
        qm_list.pop(0)
        if np.max(qm_list) - np.min(qm_list) <= converge_d:
            converged = True
            break
print(f"dQ(Robust): {round(base_Qr - Qmr_i, 2)}, Q(Robust): {round(Qmr_i, 2)}, % dQ(Robust): {100 * (round(1 - Qmr_i/base_Qr, 4))}, Q(Aux): {round(qa_list[-1], 2)}, Q(True): {round(Qm_i, 2)}, Converged: {converged}")

In [ ]:
target_q = complete_q_list

x_list = list(range(len(target_q)))
qa_values = target_q

Q_plot = go.Figure()
Q_plot.add_trace(go.Scatter(x=x_list, y=qa_values))
Q_plot.update_layout(height=800, width=800)
Q_plot.show()

In [ ]:
w_mask[1,0]

In [ ]:
# Constrained - Profiles/Contributions
factor_i = 1
c_W = W_i
c_H = H_i

b_W = nmf_model.W[:, factor_i]
b_H = nmf_model.H[factor_i]
b_H_sum = nmf_model.H.sum(axis=0)
b_factor_matrix = np.matmul(b_W.reshape(len(b_W), 1), [b_H])

b_factor_conc_sum = b_factor_matrix.sum(axis=0)
b_factor_conc_sum[b_factor_conc_sum == 0.0] = 1e-12

i_W = c_W[:, factor_i]
i_H = c_H[factor_i]
i_H_sum = c_H.sum(axis=0)
i_factor_matrix = np.matmul(i_W.reshape(len(i_W), 1), [i_H])

i_factor_conc_sum = i_factor_matrix.sum(axis=0)
i_factor_conc_sum[i_factor_conc_sum == 0] = 1e-12

b_norm_H = np.round(100 * (b_H / b_H_sum),2)
i_norm_H = np.round(100 * (i_H / i_H_sum),2)

# Highlight the factor feature elements which were in an expression.
marker_line_colors = ['rgb(125,220,157)']*len(data_handler.features)
for factor_element, fe_details in exp_mapping.items():
    if 'feature' in factor_element:
        factor_j = fe_details['index'][0]
        factor_feature_k = fe_details['index'][1]
        if factor_j == factor_i:
            marker_line_colors[factor_feature_k] = 'rgb(255,44,44)'
# Highlight the factor feature elements which were constrained
for j in range(len(h_mask[factor_i])):
    h_mask_j = h_mask[factor_i,j]
    if h_mask_j != 0:
        marker_line_colors[j] = 'rgb(122,44,255)'

fig = make_subplots(specs=[[{"secondary_y": True}]], rows=1, cols=1)
fig.add_trace(go.Scatter(x=data_handler.features, y=b_norm_H, mode="markers", marker=dict(color='gray'), name="Base % of Features", opacity=0.8), secondary_y=True, row=1, col=1)
fig.add_trace(go.Scatter(x=data_handler.features, y=i_norm_H, mode="markers", marker=dict(color='red'), name="Constrained % of Features", opacity=0.6), secondary_y=True, row=1, col=1)
fig.add_trace(go.Bar(x=data_handler.features, y=b_factor_conc_sum, marker_color='rgb(203,203,203)', marker_line_color='rgb(186,186,186)', marker_line_width=1.5, opacity=0.6, name='Base Conc. of Features'), secondary_y=False, row=1, col=1)
fig.add_trace(go.Bar(x=data_handler.features, y=i_factor_conc_sum, marker_color='rgb(134,236,168)', marker_line_color=marker_line_colors, marker_line_width=1.5, opacity=0.6, name='Constrained Conc. of Features'), secondary_y=False, row=1, col=1)
fig.update_layout(width=1200, height=600, title=f"Constrained Factor Profile - Factor {factor_i}", barmode='group', scattermode='group', hovermode="x unified")
fig.update_yaxes(type="log", secondary_y=False, range=[0, np.log10(b_factor_conc_sum).max()], row=1, col=1)
fig.update_yaxes(secondary_y=True, range=[0, 100])
fig.show()

b_norm_contr = b_W / b_W.mean()
b_data_df = cosrc.copy(data_handler.input_data)
b_data_df[factor_label] = b_norm_contr
b_data_df.index = pd.to_datetime(b_data_df.index)
b_data_df = b_data_df.sort_index()
b_data_df = b_data_df.resample('D').mean()

i_norm_contr = i_W / i_W.mean()
i_data_df = cosrc.copy(data_handler.input_data)
i_data_df[factor_label] = i_norm_contr
i_data_df.index = pd.to_datetime(i_data_df.index)
color_index = list(i_data_df.index)
i_data_df = i_data_df.sort_index()
i_data_df = i_data_df.resample('D').mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=b_data_df.index, y=b_data_df[factor_label], mode='lines+markers', marker_color='rgb(186,186,186)', name="Base Factor Contributions"))
fig.add_trace(go.Scatter(x=i_data_df.index, y=i_data_df[factor_label], mode='lines+markers', marker_color='rgb(125,220,157)', name="Constrained Factor Contributions"))
fig.update_layout(width=1200, height=800, title=f"Constrained Factor Contributions - Factor {factor_i}", hovermode="x unified")
fig.update_yaxes(title_text="Factor Contributions")
fig.show()

In [ ]:
# Constrained - Factor Fingerprints
import plotly.express as px

b_H = nmf_model.H

b_normalized = 100 * (b_H / b_H.sum(axis=0))
c_normalized = 100 * (c_H / c_H.sum(axis=0))

c_factors_fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Base Profile", "Constrained Profile"), vertical_spacing=0.075)
colors = px.colors.sequential.Viridis_r
for idx in range(factors-1, -1, -1):
    c_factors_fig.add_trace(go.Bar(name=f"Base Factor {idx+1}", x=data_handler.features, y=b_normalized[idx], marker_color=colors[idx]), row=1, col=1)
    c_factors_fig.add_trace(go.Bar(name=f"Constrained Factor {idx+1}", x=data_handler.features, y=c_normalized[idx], marker_color=colors[idx]), row=2, col=1)
c_factors_fig.update_layout(title=f"Constrained Factor Fingerprints", width=1200, height=800, barmode='stack', hovermode='x unified')
c_factors_fig.update_yaxes(title_text="% Feature Concentration", range=[0, 100])
c_factors_fig.show()

In [ ]:
import plotly.figure_factory as ff
# Constrained - G-Space Plots

b_W = nmf_model.W

b_normalized_factors_contr = b_W / b_W.sum(axis=0)
c_normalized_factors_contr = c_W / c_W.sum(axis=0)

f1_idx = 0
f2_idx = 1
show_base = True
show_delta = True

if show_delta:
    arrows = ((c_normalized_factors_contr[:, f1_idx] - b_normalized_factors_contr[:, f1_idx]), (c_normalized_factors_contr[:, f2_idx] - b_normalized_factors_contr[:, f2_idx]))
    c_g_fig = ff.create_quiver(x=b_normalized_factors_contr[:, f1_idx],y=b_normalized_factors_contr[:, f2_idx], u=arrows[0], v=arrows[1], name="Constrained Delta", line_width=1, arrow_scale=0.01, scale=0.99)
    c_g_fig.add_trace(go.Scatter(x=c_normalized_factors_contr[:, f1_idx],y=c_normalized_factors_contr[:, f2_idx], mode='markers', name="Constrained"))
    c_g_fig.add_trace(go.Scatter(x=b_normalized_factors_contr[:, f1_idx],y=b_normalized_factors_contr[:, f2_idx], mode='markers', name="Base"))
else:    
    c_g_fig = go.Figure()
    c_g_fig.add_trace(go.Scatter(x=c_normalized_factors_contr[:, f1_idx],y=c_normalized_factors_contr[:, f2_idx], mode='markers', name="Fpeak"))
    if show_base:
        c_g_fig.add_trace(go.Scatter(x=b_normalized_factors_contr[:, f1_idx],y=b_normalized_factors_contr[:, f2_idx], mode='markers', name="Base"))
c_g_fig.update_layout(title=f"Constrained G-Space Plot", width=800, height=800)
c_g_fig.update_yaxes(title_text=f"Factor {f1_idx+1} Contributions (avg=1)")
c_g_fig.update_xaxes(title_text=f"Factor {f2_idx+1} Contributions (avg=1)")
c_g_fig.show()

In [ ]:
#Constrained - Factor Contributions
contribution_threshold = 0.06
converged = True
feature_idx = 1

x_label = data_handler.input_data.columns[feature_idx]
factors_data = c_H
normalized_factors_data = 100 * (factors_data / factors_data.sum(axis=0))

feature_contr = normalized_factors_data[:, feature_idx]
feature_contr_inc = []
feature_contr_labels = []
feature_legend = {}
for idx in range(feature_contr.shape[0]-1, -1, -1):
    idx_l = idx+1
    if feature_contr[idx] > contribution_threshold:
        feature_contr_inc.append(feature_contr[idx])
        feature_contr_labels.append(f"Factor {idx_l}")
        feature_legend[f"Factor {idx_l}"] = f"Factor {idx_l} = {factors_data[idx:, feature_idx]}"
feature_fig = go.Figure(data=[go.Pie(labels=feature_contr_labels, values=feature_contr_inc, hoverinfo="label+value", textinfo="percent")])
feature_fig.update_layout(title=f"Factor Contributions to Feature: {x_label}", width=1200, height=600,
                                  legend_title_text=f"Factor Contribution > {contribution_threshold}%")
feature_fig.show()

factors_contr = c_W
normalized_factors_contr = 100 * (factors_contr / factors_contr.sum(axis=0))
factor_labels = [f"Factor {i}" for i in range(1, normalized_factors_contr.shape[1]+1)]
contr_df = pd.DataFrame(normalized_factors_contr, columns=factor_labels)
contr_df.index = pd.to_datetime(data_handler.input_data.index)
contr_df = contr_df.sort_index()
contr_df = contr_df.resample('D').mean()

contr_fig = go.Figure()
for factor in factor_labels:
    contr_fig.add_trace(go.Scatter(x=contr_df.index, y=contr_df[factor], mode='lines+markers', name=factor))
contr_fig.update_layout(title=f"Factor Contributions (avg=1)",
                                width=1200, height=600,
                                legend=dict(orientation="h", xanchor="right", yanchor="bottom", x=1, y=1.02))
contr_fig.update_yaxes(title_text="Normalized Contribution")
contr_fig.show()

### Constrained Model

The constrained model provides the functionality for a user to use their prior knowledge about the source profiles to place limits and constraints on factor elements of the solution.

The constrained model allows for two different options for placing these limits on the values of the solution, expressions and constraints. Constraints allow for a user to set precise limits on the value of a factor element, through several different constraint types. Expressions allow for the user to define how a factor element value is related to one or more other factor element values. These expressions are limited by the change in Q as a results of the new factor element values.

Both constraints and expressions are evaluated at every iteration of the constrained model run, with expressions occuring first and the constraints being applied second. Constraints are either determined exactly prior to running the model, limited by a bounds, or are calculated by a value that results in a target dQ. Expressions are dynamic as the values and sum of the expression can change from one iteration to the next.

Expressions are evaluated collectively as a set of linear equations, converted into matrix form. Such that:</br></br>
$$c1*x1 + c2*x2 = 0$$
$$c3*x3 - c4*x4 + c5*x5 = 0$$
$$c6*x2 + c7*x5 = 0$$
is equivalent to
$$\begin{bmatrix}
c1 & c2 & 0 & 0 & 0 \\
0 & 0 & c3 & -c4 & c5 \\
0 & c6 & 0 & 0 & c7 \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \\ 0 \end{bmatrix}$$ 

The matrix of expressions is evaluated at each iteration to account for updates to the factor elements, both from the constraints and the update algorithm.

A completed constrained model run will provided a constrained model which can then be used on any of the error estimation methods, by passing the .constrained_model (NMF) as the base model was originally.

In [ ]:
# Import the Constrained Model module
from esat.rotational.constrained import ConstrainedModel

constrained_model = ConstrainedModel(base_model=nmf_model, data_handler=data_handler, softness=1.0)

#### Adding Constraints
There are 6 different constraint types available:</br>
"pull down", "pull up", "pull to value", "set to zero", "set to base value", "define limits"

The target of the constraint can be either "feature" (H) or "sample" (W)

The target_value argument for a constraint takes different values depending on the constraint type:</br>
For pull down and pull up constraints, the target_value is the dQ limit of the pull</br>
For pull to value, the target_value is a tuple of (the value to pull to, the dQ limit)</br>
For define limits, the target_value is a tuple of (the min value, the max value)</br>
All other constraints do not use the target_value parameter.</br>

Constraints are added to a constrained model by the .add_constraint() method. A factor element can only have one constraint.</br>
Note that the index of the matrices are 'feature'=(index of factor, index of feature) and 'sample'=(index of sample, index of factor) 

In [ ]:
# Add a 'set to zero 'constraint on the Factor-Feature matrix (H) index (0, 3)
constrained_model.add_constraint(constraint_type="set to zero", index=(0,3), target="feature")

In [ ]:
# Add a 'define limits' constraint
constrained_model.add_constraint(constraint_type="define limits", index=(2,10), target="feature", target_values=(0.1, 0.9))

In [ ]:
# Add 'pull up' and 'pull down' constraints
constrained_model.add_constraint(constraint_type="pull up", index=(2, 15), target="feature", target_values=100)
constrained_model.add_constraint(constraint_type="pull down", index=(5, 1), target="feature", target_values=50)

In [ ]:
# Add 'set to base value' constraint
constrained_model.add_constraint(constraint_type='set to base value', index=(3, 3), target="feature")

In [ ]:
# Add 'pull to value' constraint
constrained_model.add_constraint(constraint_type='pull to value', index=(4, 20), target="feature", target_values=(2.5, 50))

In [ ]:
# Display all constraints
constrained_model.list_constraints()

In [ ]:
# Constraints can be remove before training the model with 'constraint_model.remove_constraint(constraint_label=LABEL)' where LABEL is the string 'factor:I|feature:J', also shown when the constraints are listed.
# constrained_model.remove_constraint(constraint_label='factor:2|feature:15')
constrained_model.list_constraints()

#### Adding Expressions
An expression is an equation involving 2 or more factor elements (from either W or H) where each factor element has a coefficient and the equation equals 0.</br>

A factor element is defined by the string 'factor:F|feature:K' or 'factor:F|sample:J' where F, K, J are indecies. The factor element is defined inside of [ ], such as '[factor:1|feature:1]'.</br> 

A coefficient is placed in front of the factor element, and must be present (set to 1.0 in those cases). The coefficient of a factor element is structured like '10.0*[factor:2|feature:10]'</br>
These coefficient/factor element terms are bounded by () and between multiple terms can be the +/- operator. The equations are all equal to zero.

The expression string contains two components, the expression itself and a number that specifies the dQ limit.

Examples of expression strings:

"(0.66*[factor:1|feature:2])-(4.2*[factor:2|feature:4])=0,250"</br>
"(0.35*[factor:0|feature:3])-(2.0*[factor:1|feature:3])-(3.7*[factor:3|feature:4])=0,250"</br>
"(3.2*[factor:2|feature:4])+(1.2*[factor:3|feature:10])+(0.1*[factor:1|feature:3])+(20.0*[factor:4|feature:3])-(10.7*[factor:5|feature:4])=0,250"</br>

The expressions are solved at each iteration where the equation is set to equal the sum of all factor elements. These expressions are solved as a single linear equation, with the sum of all dQ limits used to reduce the change from the original values to the solved target values.

There is currently no limit on the number of expressions or the number of times a factor element can be in an expression. A factor element can be in both a constraint and one or more expressions.

During each iteration of the constained model training method, the expressions are evaluated first and then the constraints are applied. These constraint and expressions results are used to calculate a difference matrix, which is used to modify the solution matrix. The difference matrix is the difference between the constrained target values and the current solution.

In [ ]:
# Here are three example expressions where each term is defined inside (), containing a coefficient and factor element.
expression1 = "(0.66*[factor:1|feature:2])-(4.2*[factor:2|feature:4])=0,250"
expression2 = "(0.35*[factor:0|feature:3])-(2.0*[factor:1|feature:3])-(3.7*[factor:3|feature:4])=0,250"
expression3 = "(3.2*[factor:2|feature:4])+(1.2*[factor:3|feature:10])+(0.1*[factor:1|feature:3])+(20.0*[factor:4|feature:3])-(10.7*[factor:5|feature:4])=0,250"

In [ ]:
# Expressions are added with the add_expression function
constrained_model.add_expression(expression1)
constrained_model.add_expression(expression2)
constrained_model.add_expression(expression3)

In [ ]:
# All expressions currently added for the model can be listed using the list_expressions() function
constrained_model.list_expressions()

In [ ]:
# An expression can be removed with the .remove_expression and the expression index as found in the list of 
# constrained_model.remove_expression(expression_idx=0)
constrained_model.list_expressions()

In [ ]:
# Once all the constraints and expressions have been defined and added. We can tain the constrained model.
# When the model is finished training, a new NMF model is availble in the constrained_model object as the constrained_model variable
constrained_model.train(max_iterations=5000)

In [ ]:
# Print the results of the constrained model run, as seen in the PMF5 results.
constrained_model.display_results()
# Show the plot of how Q changes during the constrained model training. Qtype can be 'True', 'Robust', 'aux'.
constrained_model.plot_Q(Qtype="Aux") # Qtype: True, Robust, Aux

In [ ]:
# Plot the profile and contribution graphs of the constrained model for a specified factor index.
constrained_model.plot_profile_contributions(factor_idx=1)

In [ ]:
# Plot the factor fingerprint stacked bar graph for the constrained model
constrained_model.plot_factor_fingerprints()

In [ ]:
# Plot the g space graph for the constrained model.
# Two optional arguments are available: show_base=Boolean, show_delta=Boolean. These default to False.
constrained_model.plot_g_space(factor_idx1=1, factor_idx2=2, show_base=True, show_delta=True)

In [ ]:
# Plot the factor contributions of the constrained model for a specified feature.
constrained_model.plot_factor_contributions(feature_idx=4)

In [ ]:
exp_elements = constrained_model.expression_mapped.keys()
constraints = constrained_model.constraints
cm_model = constrained_model.constrained_model
constraints

In [ ]:
constraint_values = []
for k, c in constraints.items():
    value = cm_model.H[c.index] if c.target == "feature" else cm_model.W[c.index]
    dQ = None
    target_value = None
    min_value = None
    max_value = None
    if c.constraint_type in ("pull down", "pull up", "pull to value"):
        if c.constraint_type == 'pull to value':
            target_value = c.target_values[0]
            dQ = c.target_values[1]
        else:
            dQ = c.target_values
    elif c.constraint_type == "define limits":
        min_value = c.target_values[0]
        max_value = c.target_values[1]
    elif c.constraint_type == "set to base value":
        target_value = constrained_model.base_model.H[c.index] if c.target == "feature" else constrained_model.base_model.W[c.index]
    constraint_values.append((c.index, c.target, c.constraint_type, target_value, value, dQ, min_value, max_value)) 

c_columns = ["index", "target", "type", "target value", "value", "dQ", "min value", "max value"]
c_df = pd.DataFrame(constraint_values, columns=c_columns)
c_df

In [ ]:
constrained_model.expression_mapped

In [ ]:
constrained_model.expression_labeled

In [ ]:
exp_element = {}
eval_exp = []
_base_model = constrained_model.base_model
for i in range(len(constrained_model.expression_labeled)):
    exp_values = []
    expression_i = constrained_model.expression_labeled[i]
    for ele in expression_i[0]:
        element_details = constrained_model.expression_mapped[ele]
        ele_index = element_details['index']
        ele_coef_idx = element_details["exp_i"].index(i)
        ele_coef = element_details["coef"][ele_coef_idx]
        ele_value = cm_model.H[ele_index] if element_details["type"] == "feature" else cm_model.W[ele_index]
        base_value = _base_model.H[ele_index] if element_details["type"] == "feature" else _base_model.W[ele_index]
        exp_element[ele] = (ele_index, element_details['type'], ele_value, base_value)
        exp_values.append(ele_value*ele_coef)
    eval_exp.append(np.sum(exp_values))
e_columns = ["index", "type", "value", "base value"]
e_df = pd.DataFrame(exp_element.values(), columns=e_columns)
e_df

In [ ]:
eval_exp